# CropForecastLK: Phase 1 — Problem Definition & Exploratory Data Analysis
**Sri Lanka Highland Crops Production Forecasting & Agricultural Intelligence System**

---

### Step 1: Problem Definition & Domain Context
Sri Lanka's highland districts (**Nuwara Eliya, Badulla, Kandy, Matale, Moneragala**) represent critical production zones for vital highland crops:
- **Kurakkan** (Finger Millet)
- **Maize**
- **Green Gram**
- **Chillies (Green)**
- **Potatoes**
- **Sweet Potatoes**
- **Manioc (Cassava)**

Agricultural yields and total harvest production in Sri Lanka fluctuate substantially across the two distinct monsoon regimes:
1. **Maha Season**: North-East Monsoon (September to March) — primary wet agricultural season with higher cultivated extent and precipitation.
2. **Yala Season**: South-West Monsoon (May to August/September) — secondary agricultural season characterized by lower rainfall and potential water stress in rain-shadow areas.

**Objective**: Formulate and evaluate a supervised regression pipeline to predict **Production (Metric Tons)** and understand yield determinants across highland agricultural zones.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

# Add ml_pipeline to sys.path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from ml_pipeline.config import RAW_DATA_PATH, HIGHLAND_DISTRICTS, TARGET_CROPS
from ml_pipeline.data_ingestion import load_raw_data, validate_schema, get_raw_summary


### Step 2: Automated Data Ingestion & Raw Schema Validation

In [ ]:
# Load raw dataset
df_raw = load_raw_data(RAW_DATA_PATH)
is_valid, audit_report = validate_schema(df_raw)

print(f"Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Schema Validation Passed: {is_valid}")
display(df_raw.head())


### Step 3: Descriptive Raw EDA & Data Type Profiling
We inspect the data types, non-null values, and identify formatting irregularities (such as thousands commas and text placeholders in numeric fields).

In [ ]:
print("--- Column Information & Data Types ---")
display(df_raw.info())

print("\n--- Missing Values Profile ---")
missing_summary = pd.DataFrame({
    'Missing Values': df_raw.isna().sum(),
    'Percentage (%)': (df_raw.isna().sum() / len(df_raw)) * 100
})
display(missing_summary)


In [ ]:
# Inspect unique categorical entities
print(f"Total Unique Districts: {df_raw['District'].nunique()}")
print(f"Unique Crop Categories: {df_raw['CropCategory'].nunique()}")
print(f"Unique Crops: {df_raw['Crop'].nunique()}")
print(f"Unique Seasons: {df_raw['Season'].unique().tolist()}")

# Check for non-district aggregate rows
print("\nSample Districts:")
print(df_raw['District'].unique()[:10])


### Step 4: Seasonal Monsoonal & Geographic EDA
Here we analyze the distribution of records across seasons and investigate the agricultural representation of key highland crops and districts.

In [ ]:
# Seasonal record distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Season counts
season_counts = df_raw['Season'].value_counts()
axes[0].bar(season_counts.index, season_counts.values, color=['#2b5c8f', '#2a9d8f', '#e76f51'], width=0.5)
axes[0].set_title("Distribution of Records by Season", fontsize=13, fontweight='bold')
axes[0].set_ylabel("Record Count")
for i, v in enumerate(season_counts.values):
    axes[0].text(i, v + 500, f"{v:,}", ha='center', fontweight='bold')

# Plot Top Crop Categories
cat_counts = df_raw['CropCategory'].value_counts().head(7)
axes[1].barh(cat_counts.index[::-1], cat_counts.values[::-1], color='#3a86ff')
axes[1].set_title("Top 7 Agricultural Crop Categories", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Record Count")

plt.tight_layout()
plt.show()


In [ ]:
# Representation of Highland Districts
highland_data = df_raw[df_raw['District'].isin(HIGHLAND_DISTRICTS)]
highland_district_counts = highland_data['District'].value_counts()

plt.figure(figsize=(10, 4))
plt.bar(highland_district_counts.index, highland_district_counts.values, color='#10b981', edgecolor='#047857', width=0.55)
plt.title("Record Counts for Target Highland Districts (2000 - 2023)", fontsize=13, fontweight='bold')
plt.ylabel("Number of Agricultural Records")
for i, v in enumerate(highland_district_counts.values):
    plt.text(i, v + 40, f"{v:,}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


### Key Findings & Pipeline Directives
1. **String Formatting**: `Extent` and `Production` columns contain strings with commas (e.g. `"4,986.0"`) and missing strings (`"-"`, `"n.a."`). A regex numeric parser is required.
2. **Aggregate Rows**: `'National Total'` is embedded in the `District` column, and `'Total'` is embedded in `Season`. These must be filtered to prevent artificial inflation and data leakage.
3. **Target Highland Crops**: Highland staples (Kurakkan, Maize, Green Gram, Chillies, Potatoes, Sweet Potatoes, Manioc) exhibit complete historical representation across 2000–2023.
4. **Next Step**: Proceed to `02_cleaning_and_feature_engineering.ipynb` for regex parsing, group median imputation, and temporal lag derivation.
